In [8]:
import requests
import pandas as pd
import datetime 
from datetime import datetime, UTC
from __future__ import annotations
import math
from typing import Any, Optional

In [ ]:
import sys
sys.path.append("../src")

from config import AIRLINES_TABLE_ID, AIRLINE_API_URL, HEADERS, DEFAULT_API_TIMEOUT
from bq import load_df_to_bq

In [10]:
def transform(records: list[dict]) -> pd.DataFrame:
    rows = []
    collected_at = datetime.now(UTC)

    for r in records:
        row = {
            "airline_icao": r.get("icaoCode"),
            "airline_iata": r.get("iataCode"),

            "airline_kr": r.get("name2"),
            "airline_en": r.get("name"),

            "city_kr": r.get("koCity"),
            "city_en": r.get("enCity"),

            "country_kr": r.get("koCountry"),
            "country_en": r.get("en2Country"),

            "continent_kr": r.get("areaKr"),
            "continent_en": r.get("areaEn"),
            
            "business_model": r.get("businessModel"),
            "operating_status": r.get("section"),
                        
            "collected_at": collected_at,
        }
        rows.append(row)
        
    return pd.DataFrame(rows)

In [ ]:
def _fetch_airlines_one(active: Optional[bool], page_size: int = 100) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []

    payload: dict[str, Any] = {
        "pageNumber": 1,
        "pageSize": page_size,
        "order": "asc",
        "orderNm": "name2",
    }
    if active is not None:
        payload["active"] = active

    resp = requests.post(AIRLINE_API_URL, json=payload, headers=HEADERS, timeout=DEFAULT_API_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()

    content = data.get("content") or []
    total = data.get("total")

    rows.extend(content)

    total_pages = math.ceil(total / page_size)

    for page in range(2, total_pages + 1):
        payload["pageNumber"] = page
        resp = requests.post(AIRLINE_API_URL, json=payload, headers=HEADERS, timeout=DEFAULT_API_TIMEOUT)
        resp.raise_for_status()
        d = resp.json()
        rows.extend(d.get("content") or [])

    return rows

In [12]:
def fetch_airlines_all(page_size: int = 100) -> list[dict[str, Any]]:
    active_rows = _fetch_airlines_one(active=True, page_size=page_size)
    inactive_rows = _fetch_airlines_one(active=False, page_size=page_size)

    merged: dict[str, dict[str, Any]] = {}
    for r in active_rows + inactive_rows:
        key = r.get("icaoCode") 
        merged[key] = r

    return list(merged.values())

In [13]:
def run_airlines_pipeline() -> None:
    airline_records = fetch_airlines_all()
    if not airline_records:
        raise RuntimeError("[AIRLINES] fetched 0 records")

    airline_df = transform(airline_records)
    if airline_df.empty:
        raise RuntimeError("[AIRLINES] transformed 0 rows")

    if "airline_icao" in airline_df.columns:
        airline_df = airline_df.drop_duplicates(subset=["airline_icao"], keep="last")

    load_df_to_bq(airline_df, table_id=AIRLINES_TABLE_ID)
    print(f"[AIRLINES] loaded {len(airline_df)} rows to BigQuery")

In [14]:
try:
    run_airlines_pipeline()
except Exception as e:
    print(f"[AIRLINES] pipeline failed: {e}")
    raise

[AIRLINES] loaded 1941 rows to BigQuery
